## Scope Connect and Setup

In [1]:
import chipwhisperer as cw
scope = cw.scope()
scope.default_setup()

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 327828                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.clock.freq_ctr                     changed from 0                         to 1054481                  
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

## Target Connect

In [2]:
target = cw.target(scope)

## Object Info

In [3]:
print(scope) # also works with project, CPA, etc.

cwhuskyplus Device
sn             = 50203220573555303130353137323034
fpga_buildtime = 12/16/2024, 13:17
fw_version = 
    major = 1
    minor = 2
    debug = 0
gain = 
    mode = high
    gain = 22
    db   = 25.091743119266056
adc = 
    state                    = False
    basic_mode               = rising_edge
    timeout                  = 2
    offset                   = 0
    presamples               = 0
    samples                  = 5000
    decimate                 = 1
    trig_count               = 31612
    stream_mode              = False
    test_mode                = False
    bits_per_sample          = 12
    segments                 = 1
    segment_cycles           = 0
    segment_cycle_counter_en = False
    clip_errors_disabled     = False
    lo_gain_errors_disabled  = False
    errors                   = False
clock = 
    clkgen_src             = system
    clkgen_freq            = 7363636.363636363
    adc_mul                = 4
    adc_freq               = 294545

## Build Firmware

In [4]:
%%bash
cd ../firmware/mcu/simpleserial-aes
make -j PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C

Building for platform CWHUSKY with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
.
Compiling:
Compiling:
-en     simpleserial-aes.c ...
Compiling:
.
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
-en     .././hal//sam4s/startup_sam4s.c ...
Compiling:
-en     .././hal//sam4s/sam4s_hal.c ...
.
.
Compiling:
Compiling:
.
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/pio.c ...
.
Compiling:
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/sysclk.c ...
.
.
Compiling:
Compiling:
-en     .././hal//sam4s/pmc.c .

## Program Target

In [5]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

## Capture Loop

In [6]:
from tqdm.notebook import tqdm

project = cw.Project()

N = 100
for key, text, n in tqdm(cw.ktp.BasicIt(N), desc='Capturing traces'):
    container = cw.capture_trace(scope, target, text, key)
    if container is not None:
        project.append(container)

Capturing traces:   0%|          | 0/100 [00:00<?, ?it/s]

## Save Project to Zip

In [8]:
project.save("/home/peter/test-project.zip")

saving to test-project


## Project Data Access

In [9]:
project.traces[:] # traces
project.plaintexts[:] # plaintexts
project.ciphertexts[:] # ciphertexets
project.keys[:] # keys
project[:] # project as trace container

[trace: [1658  691 1506 ... 2470 2398 2511]
 plaintext: [ 81 123 194 141  46 174 253 136  77  51 132  99  40  47  27  92]
 ciphertext: [149 243 166   4  23 105 164 245 226 142 169 230  15 167  83  20]
 key: [ 43 126  21  22  40 174 210 166 171 247  21 136   9 207  79  60],
 trace: [1674  627 1550 ... 2473 2393 2495]
 plaintext: [ 48 234 140 168 229 134 244 186 186 246 222   9 189 112  47 225]
 ciphertext: [122 218  66 245  51  52  31  75 143  45 195 156  95 165  27 249]
 key: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0],
 trace: [1640  654 1491 ... 2467 2392 2492]
 plaintext: [ 13  60 163 118  57 164  37 164  76 139  75 108  21 237 221 105]
 ciphertext: [172 251 197 190 114 254 156  10 242 192 167 179  75 109 135 105]
 key: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0],
 trace: [1664  630 1556 ... 2465 2393 2503]
 plaintext: [ 49  55  57   1  87  30 251 246  13  81 206 199  11  23 192 211]
 ciphertext: [  8 235 121 160 238  11   6  95 120  43 188 148  46 142  97 201]
 key: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0],


## Capture Loop with Real Time Plotting

In [10]:
plt = cw.StreamPlot()
plt.plot()

:DynamicMap   []
   :Curve   [x]   (y)

In [11]:
from tqdm.notebook import tqdm

project = cw.Project()

N = 100
for key, text, n in tqdm(cw.ktp.BasicIt(N), desc='Capturing traces'):
    container = cw.capture_trace(scope, target, text, key)
    if container is not None:
        project.append(container)
        plt.update(container.trace)

Capturing traces:   0%|          | 0/100 [00:00<?, ?it/s]

## Basic Attack Using Analyzer

In [12]:
from chipwhisperer.analyzer import CPA, get_table_cb, leakage_models

cb = get_table_cb()
cpa = CPA(project, leakage_models.sbox_output, 16)
cpa.run(10, callback=cb)
print(cpa.key_guess())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
PGE=,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,2B-0.795,7E-0.836,15-0.843,16-0.790,28-0.652,AE-0.741,D2-0.818,A6-0.852,AB-0.766,F7-0.821,15-0.853,88-0.877,09-0.797,CF-0.838,4F-0.852,3C-0.743
1,49-0.463,A50.483,880.484,07-0.460,B9-0.480,550.479,C1-0.490,1E-0.454,34-0.483,170.518,06-0.462,050.531,EF0.439,9E0.493,F5-0.484,A6-0.478
2,FC0.456,570.477,CF-0.455,030.454,1B-0.467,1A0.456,95-0.445,560.453,C60.483,1A0.507,680.457,960.469,950.435,5F-0.461,F90.462,A70.471
3,F7-0.455,93-0.468,9C-0.449,76-0.453,26-0.453,1F-0.449,AD-0.442,F40.444,3D0.454,A3-0.475,600.441,070.464,FA-0.433,45-0.460,5A-0.448,DC0.457
4,8B-0.449,4A0.450,A1-0.448,15-0.448,9F0.443,E00.443,1C-0.437,B40.441,39-0.444,2B-0.451,9C-0.440,16-0.463,FC-0.432,ED-0.451,890.442,54-0.455


[ 43 126  21  22  40 174 210 166 171 247  21 136   9 207  79  60]


## Plotting Analyzer Info

In [13]:
cpa.corr_v_time_plot() # or pge_v_traces_plot() or corr_v_traces_plot()

:Overlay
   .Curve.I         :Curve   [x]   (y)
   .Curve.II        :Curve   [x]   (y)
   .Curve.III       :Curve   [x]   (y)
   .Curve.Subkey_0  :Curve   [x]   (y)
   .Curve.IV        :Curve   [x]   (y)
   .Curve.V         :Curve   [x]   (y)
   .Curve.Subkey_1  :Curve   [x]   (y)
   .Curve.VI        :Curve   [x]   (y)
   .Curve.VII       :Curve   [x]   (y)
   .Curve.Subkey_2  :Curve   [x]   (y)
   .Curve.VIII      :Curve   [x]   (y)
   .Curve.IX        :Curve   [x]   (y)
   .Curve.Subkey_3  :Curve   [x]   (y)
   .Curve.X         :Curve   [x]   (y)
   .Curve.XI        :Curve   [x]   (y)
   .Curve.Subkey_4  :Curve   [x]   (y)
   .Curve.XII       :Curve   [x]   (y)
   .Curve.XIII      :Curve   [x]   (y)
   .Curve.Subkey_5  :Curve   [x]   (y)
   .Curve.XIV       :Curve   [x]   (y)
   .Curve.XV        :Curve   [x]   (y)
   .Curve.Subkey_6  :Curve   [x]   (y)
   .Curve.XVI       :Curve   [x]   (y)
   .Curve.XVII      :Curve   [x]   (y)
   .Curve.Subkey_7  :Curve   [x]   (y)
   .Curve.XVIII     :Curve   [x]   (y)
   .Curve.XIX       :Curve   [x]   (y)
   .Curve.Subkey_8  :Curve   [x]   (y)
   .Curve.XX        :Curve   [x]   (y)
   .Curve.XXI       :Curve   [x]   (y)
   .Curve.Subkey_9  :Curve   [x]   (y)
   .Curve.XXII      :Curve   [x]   (y)
   .Curve.XXIII     :Curve   [x]   (y)
   .Curve.Subkey_10 :Curve   [x]   (y)
   .Curve.XXIV      :Curve   [x]   (y)
   .Curve.XXV       :Curve   [x]   (y)
   .Curve.Subkey_11 :Curve   [x]   (y)
   .Curve.XXVI      :Curve   [x]   (y)
   .Curve.XXVII     :Curve   [x]   (y)
   .Curve.Subkey_12 :Curve   [x]   (y)
   .Curve.XXVIII    :Curve   [x]   (y)
   .Curve.XXIX      :Curve   [x]   (y)
   .Curve.Subkey_13 :Curve   [x]   (y)
   .Curve.XXX       :Curve   [x]   (y)
   .Curve.XXXI      :Curve   [x]   (y)
   .Curve.Subkey_14 :Curve   [x]   (y)
   .Curve.XXXII     :Curve   [x]   (y)
   .Curve.XXXIII    :Curve   [x]   (y)
   .Curve.Subkey_15 :Curve   [x]   (y)